# Ponzi — RF: released-split X14 baseline

**One fixed configuration, one released partition, five model seeds. No search, no resplitting.**
Only `dataRaw` and `dataPhish`/`dataHEIG` inputs are required. Run this notebook independently; no notebook00 or code/model input.

Configuration is inherited from the accepted group study, not a recovered winner of the original paper table.
Meta-IFD follows the public model/decoder and code loss with explicit runner corrections. GAT/GraphSAGE use the group's audited newer backbones;
their original-paper implementations are not fully specified. LR/RF are supplementary baselines; no paper reference score is invented.

Released Phish masks retain a known conflicting-label address across TRAIN/TEST. Its rows are kept and the overlap is reported.
This reference experiment must remain separate from the address-disjoint SIFE study. Both datasets are static/transductive.


## Environment

Install only missing packages. GNNs use GPU when available; tabular models use CPU. Package versions are frozen with each run.

In [ ]:
import importlib.util
import subprocess
import sys
INSTALL_MISSING_PACKAGES=True
requirements={'numpy':'numpy>=1.24,<3','pandas':'pandas>=2,<4','scipy':'scipy>=1.11,<2',
              'sklearn':'scikit-learn>=1.3,<2','joblib':'joblib>=1.3,<2',
              'threadpoolctl':'threadpoolctl>=3,<4','torch':'torch>=2.1',
              'torch_geometric':'torch-geometric>=2.4,<3'}
for module,requirement in requirements.items():
    if importlib.util.find_spec(module) is None:
        if not INSTALL_MISSING_PACKAGES:raise RuntimeError(f'Install {requirement} first')
        subprocess.check_call([sys.executable,'-m','pip','install',requirement])
import torch
print('Python',sys.version.split()[0],'| Torch',torch.__version__)


## Configuration

Full runs five seeds12–16 on the original masks. Smoke runs one seed and two neural epochs in its own output folder; it is not the paper table.

In [ ]:
import os
from pathlib import Path
DATASET='Ponzi'
MODEL='rf'
RUN_MODE='full'
INPUT_ROOT=Path('/kaggle/input') if Path('/kaggle/input').is_dir() else Path.cwd()
DATA_RAW=Path(os.environ.get('BASELINE_RAW_INPUT',str(INPUT_ROOT)))
DATA_PHISH=Path(os.environ.get('BASELINE_PROCESSED_INPUT',str(INPUT_ROOT)))
OUTPUT_ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORK_ROOT=Path(os.environ.get('BASELINE_WORK_ROOT',str(OUTPUT_ROOT/'paper_baseline_native_fixed_v2')))/DATASET/MODEL
DATA_OUTPUT=WORK_ROOT/'data'
JOB_DIR=WORK_ROOT/'results'/RUN_MODE
CPU_THREADS=2
DEVICE='cpu' if MODEL in ('lr','rf','lightgbm') else ('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Dataset/model:',DATASET,MODEL,'| mode:',RUN_MODE,'| device:',DEVICE)
print('Output:',WORK_ROOT)


## Implementation: utilities

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.imports
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import torch


### require

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.require
def require(condition, message):
    if not condition:
        raise ValueError(message)


### sha256

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.sha256
def sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()


### json_hash

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.json_hash
def json_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


### write_json

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.write_json
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


### read_json

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.read_json
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


### seed_everything

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.seed_everything
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### runtime_information

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.runtime_information
def runtime_information(device):
    packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'torch-geometric','joblib','threadpoolctl']
    versions = {name: importlib.metadata.version(name) for name in packages}
    try:
        versions['lightgbm'] = importlib.metadata.version('lightgbm')
    except importlib.metadata.PackageNotFoundError:
        versions['lightgbm'] = None
    for name in ['pyg-lib','torch-sparse','torch-scatter']:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return dict(python=platform.python_version(), platform=platform.platform(),
                packages=versions,
                device=str(device), cuda=torch.version.cuda,
                cpu_threads=torch.get_num_threads(),
                gpu=torch.cuda.get_device_name(device) if str(device).startswith('cuda') else None,
                sampler='pyg_neighbor_loader_directional_v1',
                bitwise_cross_platform_equivalence_claimed=False)


### seal_files

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.seal_files
def seal_files(root, files):
    root = Path(root)
    return {Path(p).relative_to(root).as_posix(): sha256(p) for p in files}


### verify_files

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.verify_files
def verify_files(root, manifest):
    for relative, expected in manifest.items():
        require(sha256(Path(root)/relative) == expected, f'Frozen file changed: {relative}')


### atomic_torch_save

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: utilities.atomic_torch_save
def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.partial')
    torch.save(value, temporary)
    os.replace(temporary, path)


## Implementation: data

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: data.imports
from contextlib import contextmanager
import hashlib
import io
import os
from pathlib import Path
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch


### resolve_input

Locate one raw ZIP/folder or one processed graph/HEIG.zip.

In [ ]:
# BASELINE_SOURCE_CELL: data.resolve_input
def resolve_input(root,dataset,kind):
    """Locate one raw ZIP/folder or one processed graph/HEIG.zip."""
    root=Path(root).expanduser().resolve()
    require(root.exists(),f'Missing {kind} input: {root}')
    if root.is_file():return root
    if kind=='raw':
        matches=list(root.rglob('OneDrive_2026-09-18.zip'))
        if len(matches)==1:return matches[0]
        marker='EOA_labeled.csv' if dataset=='Phish' else 'CA_labeled.csv'
        matches=[p.parent.parent for p in root.rglob(marker) if p.parent.name==dataset+'_unbalance']
    else:
        matches=[p for p in root.rglob('data.pt') if dataset.lower() in [x.lower() for x in p.parts[:-1]]]
        if len(matches)==1:return matches[0]
        matches=list(root.rglob('HEIG.zip'))
    require(len(matches)==1,f'Set the exact {dataset} {kind} path in Configuration; found {matches}')
    return matches[0]


### open_raw_member

Read the same labelled table from a ZIP or extracted raw folder.

In [ ]:
# BASELINE_SOURCE_CELL: data.open_raw_member
@contextmanager
def open_raw_member(source,member):
    """Read the same labelled table from a ZIP or extracted raw folder."""
    source=Path(source)
    if source.is_file():
        with zipfile.ZipFile(source) as archive:
            with archive.open(member) as stream:yield stream
    else:
        require(member.startswith('Meta-IFD data/'),'Invalid raw member')
        path=(source/member[len('Meta-IFD data/'):]).resolve()
        require(path.is_relative_to(source.resolve()),'Unsafe raw path')
        with path.open('rb') as stream:yield stream


### obtain_graph

Hash-check the known graph before deserialization; direct files remain read-only.

In [ ]:
# BASELINE_SOURCE_CELL: data.obtain_graph
def obtain_graph(source,output,dataset,expected_hash):
    """Hash-check the known graph before deserialization; direct files remain read-only."""
    source=Path(source)
    if source.suffix.lower()=='.pt':path=source
    else:
        path=Path(output)/'graph/data.pt'
        if not path.exists():
            path.parent.mkdir(parents=True,exist_ok=True)
            temporary=path.with_suffix('.partial')
            with zipfile.ZipFile(source) as archive:
                names=[n for n in archive.namelist() if n.endswith(f'{dataset}/processed/data.pt')]
                require(len(names)==1,'Expected one target processed graph in ZIP')
                with archive.open(names[0]) as stream,temporary.open('wb') as dest:
                    shutil.copyfileobj(stream,dest,8*1024*1024)
            require(sha256(temporary)==expected_hash,'Wrong extracted graph bytes')
            os.replace(temporary,path)
    require(sha256(path)==expected_hash,'Wrong graph release; do not deserialize it')
    return path


### released_membership

Use graph train_mask/val_mask/test_mask verbatim and disclose address overlap.

In [ ]:
# BASELINE_SOURCE_CELL: data.released_membership
def released_membership(graph,reference,dataset):
    """Use graph train_mask/val_mask/test_mask verbatim and disclose address overlap."""
    target='EOA' if dataset=='Phish' else 'CA'
    ids=np.asarray(reference['ids'],dtype=np.int64)
    labels=np.asarray(reference['labels'],dtype=np.int64)
    require(np.array_equal(graph[target].y[torch.from_numpy(ids)].numpy(),labels),'Release label alignment changed')
    cohort=pd.DataFrame(dict(node_id=ids,label=labels,address=reference['addresses'])).set_index('node_id')
    parts={}
    for role,mask in [('train','train_mask'),('validation','val_mask'),('test','test_mask')]:
        selected=torch.where(graph[target][mask])[0].numpy()
        require(set(selected)<=set(ids),'Mask includes unknown labelled IDs')
        parts[role]=cohort.loc[selected].reset_index()
    combined=np.concatenate([p.node_id.to_numpy() for p in parts.values()])
    require(len(combined)==len(set(combined))==len(ids) and set(combined)==set(ids),'Released masks overlap or miss roots')
    address_sets={r:set(p.address) for r,p in parts.items()}
    overlaps={a+'_'+b:sorted(address_sets[a]&address_sets[b]) for a,b in
              [('train','validation'),('train','test'),('validation','test')]}
    expected={'Phish':{'train':1657,'validation':553,'test':553},
              'Ponzi':{'train':804,'validation':269,'test':269}}
    sizes={r:len(p) for r,p in parts.items()}
    require(sizes==expected[dataset],'Unexpected released split sizes')
    return parts,dict(dataset=dataset,target=target,rule='unchanged released masks; no resplitting',
        sizes=sizes,positives={r:int(p.label.sum()) for r,p in parts.items()},
        address_intersections=overlaps,unique_addresses=int(cohort.address.nunique()),
        removed_ids=[],exact_paper_table_run_recovered=False)


### prepare_release

Verify two inputs and source14; no incident features, raw edge restoration or fitting.

In [ ]:
# BASELINE_SOURCE_CELL: data.prepare_release
def prepare_release(raw_source,processed_source,output,dataset,reference):
    """Verify two inputs and source14; no incident features, raw edge restoration or fitting."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    path=obtain_graph(processed_source,output,dataset,reference['graph_sha256'])
    member=f'Meta-IFD data/{dataset}_unbalance/'+('EOA_labeled.csv' if dataset=='Phish' else 'CA_labeled.csv')
    with open_raw_member(raw_source,member) as stream:content=stream.read()
    expected=reference['raw_members'][member]
    require(len(content)==expected['bytes'] and hashlib.sha256(content).hexdigest()==expected['sha256'],
            'Raw labelled table checksum mismatch')
    frame=pd.read_csv(io.BytesIO(content))
    addresses=frame['name' if dataset=='Phish' else 'Address'].astype(str).str.lower().to_numpy()
    require(set(reference['addresses'])<=set(addresses),'Mapped roots missing in raw table')
    if dataset=='Phish':
        groups={1:set(addresses[:1207]),0:set(addresses[1207:])}
        require(all(a in groups[int(y)] for a,y in zip(reference['addresses'],reference['labels'])),
                'Phish label groups differ from audited mapping')
    payload=torch.load(str(path),map_location='cpu',mmap=True,weights_only=False)
    graph=payload[0] if isinstance(payload,tuple) else payload
    require(set(graph.node_types)=={'EOA','CA'},'Unexpected node types')
    require(all(graph[nt].x.shape[1]==14 for nt in graph.node_types),'Source width must be14')
    membership,report=released_membership(graph,reference,dataset)
    manifest=dict(dataset=dataset,graph_sha256=reference['graph_sha256'],raw_label_sha256=expected['sha256'],
                  address_mapping_reference_sha256=json_hash(reference),source14='unchanged released graph values',
                  mapping_scope='audited labelled-root mapping bound to known graph and raw table bytes; not rebuilding all raw nodes',
                  split_report=report)
    write_json(output/'data_manifest.json',manifest)
    for role,part in membership.items():part.to_csv(output/(role+'_membership.csv'),index=False)
    return graph,membership,manifest


## Implementation: sampling

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: sampling.imports
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
import torch_geometric.typing


### save_pyg_role_batches

Cache actual PyG NeighborLoader samples; root IDs precede neighbors, as in author code.

In [ ]:
# BASELINE_SOURCE_CELL: sampling.save_pyg_role_batches
def save_pyg_role_batches(graph, membership, target_type, role, output_dir,
                          sampler_seed, fanouts=(100,100), batch_size=512):
    """Cache actual PyG NeighborLoader samples; root IDs precede neighbors, as in author code."""
    require(torch_geometric.typing.WITH_PYG_LIB or torch_geometric.typing.WITH_TORCH_SPARSE,
            'NeighborLoader needs a working pyg-lib or torch-sparse installation for this Torch build; no sampler fallback is allowed')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Sampling needs topology and node counts, not full features or edge attributes.
    topology = HeteroData()
    for nt in graph.node_types:
        topology[nt].num_nodes = graph[nt].num_nodes
    for et in graph.edge_types:
        topology[et].edge_index = graph[et].edge_index
    roots = torch.tensor(membership.node_id.to_numpy(copy=True), dtype=torch.long)
    records = []
    # Explicit per-role seeds are a reproducibility correction, not recovered author RNG.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sampler_seed)
        loader = NeighborLoader(topology, num_neighbors=list(fanouts),
                    input_nodes=(target_type, roots), batch_size=batch_size,
                    shuffle=role != 'test', replace=False, subgraph_type='directional',
                    num_workers=0, filter_per_worker=False)
        for sampled in loader:
            size = int(sampled[target_type].batch_size)
            ids = {nt: sampled[nt].n_id.cpu().clone() for nt in graph.node_types}
            seed_ids = ids[target_type][:size]
            batch = dict(node_ids=ids,
                         edge_index={et: sampled[et].edge_index.cpu().clone() for et in graph.edge_types},
                         seed_ids=seed_ids, seed_labels=graph[target_type].y[seed_ids].cpu().clone(),
                         target_type=target_type)
            require(torch.equal(roots[sampled[target_type].input_id], seed_ids), 'PyG input/root identity mismatch')
            path = output_dir/f'batch_{len(records):04d}.pt'
            atomic_torch_save(batch, path)
            records.append(dict(file=path.name, sha256=sha256(path), seeds=seed_ids.tolist()))
    actual = [node for record in records for node in record['seeds']]
    require(len(actual) == len(set(actual)) == len(membership) and set(actual) == set(membership.node_id),
            'PyG sampled root coverage mismatch')
    write_json(output_dir/'index.json', dict(role=role, seed=sampler_seed,
        backend='pyg_neighbor_loader_directional_v1', fanouts=list(fanouts),
        batch_size=batch_size, records=records))
    return load_role_batches(output_dir)


### load_role_batches

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: sampling.load_role_batches
def load_role_batches(output_dir):
    output_dir = Path(output_dir)
    index = read_json(output_dir/'index.json')
    records = []
    for row in index['records']:
        path = output_dir/row['file']
        require(sha256(path) == row['sha256'], 'Sampled topology cache changed')
        records.append(dict(row, path=str(path)))
    return records


## Implementation: attention_conv

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: attention_conv.imports
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import reset


### my_conv

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: attention_conv.my_conv
class my_conv(nn.Module):
    def __init__(self, in_channels, out_channels, metadata, group='sum'):
        super().__init__()
        if not isinstance(in_channels, dict):
            in_channels = {nt: in_channels for nt in metadata[0]}
        if group != 'sum':
            raise ValueError('The released model uses relation sum aggregation')
        self.out_channels = out_channels
        self.v_lin = nn.ModuleDict({nt: Linear(width, out_channels) for nt,width in in_channels.items()})
        self.a_lin = nn.ModuleDict({nt: Linear(width+out_channels, out_channels) for nt,width in in_channels.items()})
        self.skip = nn.ParameterDict({nt: nn.Parameter(torch.ones(1)) for nt in in_channels})

    def reset_parameters(self):
        reset(self.v_lin)
        reset(self.a_lin)
        for parameter in self.skip.values():
            nn.init.ones_(parameter)

    def forward(self, x_dict, edge_index_dict):
        projected = {nt: self.v_lin[nt](values) for nt,values in x_dict.items()}
        incoming = {nt: [] for nt in x_dict}
        for (src, _, dst), edges in edge_index_dict.items():
            summed = projected[dst].new_zeros((len(projected[dst]), self.out_channels))
            summed.index_add_(0, edges[1], projected[src][edges[0]])
            incoming[dst].append(summed)
        output = {}
        for nt, values in x_dict.items():
            total = torch.stack(incoming[nt], dim=0).sum(dim=0)
            output[nt] = self.a_lin[nt](torch.tanh(torch.cat([total, values], dim=1)))
        return output


## Implementation: models_meta

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: models_meta.imports
from typing import Optional, List
import torch
import torch.nn as nn
from torch import Tensor
from torch_geometric.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn.inits import glorot, reset
from torch_geometric.nn import MessagePassing


### group

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: models_meta.group
def group(xs: List[Tensor], aggr: Optional[str]) -> Optional[Tensor]:
    if len(xs) == 0:
        return None
    elif aggr is None:
        return torch.stack(xs, dim=1)
    elif len(xs) == 1:
        return xs[0]
    elif aggr == "cat":
        return torch.cat(xs, dim=-1)
    else:
        out = torch.stack(xs, dim=0)
        out = getattr(torch, aggr)(out, dim=0)
        out = out[0] if isinstance(out, tuple) else out
        return out


### TripletLoss

Compute normal triplet loss or soft margin triplet loss given triplets

In [ ]:
# BASELINE_SOURCE_CELL: models_meta.TripletLoss
class TripletLoss(nn.Module):
    '''
    Compute normal triplet loss or soft margin triplet loss given triplets
    '''

    def __init__(self, margin=None):
        super(TripletLoss, self).__init__()
        self.margin = margin
        if self.margin is None:  # if no margin assigned, use soft-margin
            self.Loss = nn.SoftMarginLoss()
        else:
            self.Loss = nn.TripletMarginLoss(margin=margin, p=2)

    def forward(self, anchor, pos, neg):
        if self.margin is None:
            num_samples = anchor.shape[0]
            y = torch.ones((num_samples, 1)).view(-1)
            if anchor.is_cuda: y = y.cuda()
            ap_dist = torch.norm(anchor - pos, 2, dim=1).view(-1)
            an_dist = torch.norm(anchor - neg, 2, dim=1).view(-1)
            loss = self.Loss(an_dist - ap_dist, y)
        else:
            loss = self.Loss(anchor, pos, neg)

        return loss


### HMSL

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: models_meta.HMSL
class HMSL(nn.Module):
    def __init__(self, hidden, out_channels, data, concat, target_node, input_width=14):
        super().__init__()
        self.concat_num = concat
        self.target = target_node
        self.loss_co = TripletLoss(margin=0.3)

        self.lin_dict = torch.nn.ModuleDict()
        self.lin_dict_mean = torch.nn.ModuleDict()
        for node_type in data.node_types:
            self.lin_dict[node_type] = Linear(input_width, hidden)
            self.lin_dict_mean[node_type] = Linear(hidden, hidden)
        self.lin_out = Linear(hidden, out_channels)

        self.k_lin = torch.nn.ModuleDict()
        self.q_lin = torch.nn.ModuleDict()
        self.v_lin = torch.nn.ModuleDict()

        for node_type in data.node_types:
            self.k_lin[node_type] = Linear(hidden, hidden)
            self.q_lin[node_type] = Linear(hidden, hidden)
            self.v_lin[node_type] = Linear(hidden, hidden)

        self.conv = my_conv(hidden, hidden, metadata=data.metadata())
        self.conv1 = my_conv(hidden, hidden, metadata=data.metadata())

    def reset_parameters(self):
        reset(self.lin_dict)
        reset(self.lin_dict_mean)
        reset(self.k_lin)
        reset(self.q_lin)
        reset(self.v_lin)
        reset(self.conv)
        reset(self.conv1)
        self.lin_out.reset_parameters()

    def forward(self, x_dict, edge_index):
        CA_hidden_ls = []
        EOA_hidden_ls = []

        ca_out_ls = []
        eoa_out_ls = []
        out_dict = {}
        for k in range(self.concat_num):
            x_dict_ = {}
            for node_type, x in x_dict.items():
                layer = self.lin_dict[node_type]
                encoded = layer(x[k])
                x_dict_[node_type] = torch.tanh(encoded)
            CA_hidden_ls.append(x_dict_['CA'])
            EOA_hidden_ls.append(x_dict_['EOA'])
            CA_h = torch.mean(CA_hidden_ls[k], dim=0)
            CA_h = F.tanh(self.lin_dict_mean['CA'](CA_h))
            EOA_h = torch.mean(EOA_hidden_ls[k], dim=0)
            EOA_h = F.tanh(self.lin_dict_mean['EOA'](EOA_h))
            ca_out_ls.append(CA_h)
            eoa_out_ls.append(EOA_h)

        out_dict['CA'] = ca_out_ls
        out_dict['EOA'] = eoa_out_ls

        for node_type, x in out_dict.items():
            out_dict[node_type] = self.attention(x, node_type)

        out_dict = self.conv(out_dict, edge_index)
        out_dict = self.conv1(out_dict, edge_index)

        out = self.lin_out(out_dict[self.target])
        loss = self.contrast_module(CA_hidden_ls, EOA_hidden_ls)

        return out, loss

    def contrast_module(self, CA_hidden_ls, EOA_hidden_ls):
        anchors = []
        pos = []
        neg = []
        for i, z in enumerate(CA_hidden_ls):
            anchors.append(z[1])
            pos.append(z[2])
            neg.append(z[0])

        out_anchors = self.attention(anchors, 'CA')
        out_pos = self.attention(pos, 'CA')
        out_neg = self.attention(neg, 'CA')
        loss_ca = self.loss_co(out_anchors, out_pos, out_neg)

        anchors = []
        pos = []
        neg = []
        for i, z in enumerate(EOA_hidden_ls):
            anchors.append(z[1])
            pos.append(z[2])
            neg.append(z[0])

        out_anchors = self.attention(anchors, 'EOA')
        out_pos = self.attention(pos, 'EOA')
        out_neg = self.attention(neg, 'EOA')
        loss_eoa = self.loss_co(out_anchors, out_pos, out_neg)
        loss = loss_ca + loss_eoa
        return loss

    def attention(self, input, node_type):
        input = torch.stack(input, dim=0).permute(1, 0, 2)
        q = self.q_lin[node_type](input)
        k = self.k_lin[node_type](input)
        v = self.v_lin[node_type](input)
        att = torch.matmul(k, q.transpose(1, 2))
        att = torch.nn.functional.softmax(att, dim=-1)
        v = att @ v
        v = torch.mean(v, dim=1)
        return v


## Implementation: models

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: models.imports
import hashlib
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv,SAGEConv


### SignedLogGAT

Audited GAT: signed-log/TRAIN z-score -> native4-head GAT -> ELU -> GAT2.

In [ ]:
# BASELINE_SOURCE_CELL: models.SignedLogGAT
class SignedLogGAT(nn.Module):
    """Audited GAT: signed-log/TRAIN z-score -> native4-head GAT -> ELU -> GAT2."""
    def __init__(self,hidden,statistics=None):
        super().__init__();self.input_width=14
        mean=torch.zeros(14) if statistics is None else torch.tensor(statistics['mean'],dtype=torch.float32)
        scale=torch.ones(14) if statistics is None else torch.tensor(statistics['scale'],dtype=torch.float32)
        require(mean.shape==scale.shape==(14,) and bool(torch.isfinite(mean).all()) and
                bool(torch.isfinite(scale).all() and (scale>0).all()),'Invalid frozen GAT normalization')
        self.register_buffer('input_mean',mean);self.register_buffer('input_scale',scale)
        self.conv1=GATConv(14,hidden,heads=4,concat=True,dropout=0.,negative_slope=.2,add_self_loops=True,bias=True)
        self.conv2=GATConv(4*hidden,2,heads=4,concat=False,dropout=0.,negative_slope=.2,add_self_loops=True,bias=True)
    def normalize_input(self,x):
        require(x.ndim==2 and x.shape[1]==14,'Expected unchanged source14')
        transformed=x.sign()*x.abs().log1p()
        return (transformed-self.input_mean)/self.input_scale
    def forward(self,x,edge_index):
        h=self.normalize_input(x)
        return self.conv2(F.elu(self.conv1(h,edge_index)),edge_index)


### InputEncodedSAGE

Audited SAGE: rawX14 -> Linear/tanh -> mean SAGE(project=True) -> ReLU -> SAGE2.

In [ ]:
# BASELINE_SOURCE_CELL: models.InputEncodedSAGE
class InputEncodedSAGE(nn.Module):
    """Audited SAGE: rawX14 -> Linear/tanh -> mean SAGE(project=True) -> ReLU -> SAGE2."""
    def __init__(self,hidden):
        super().__init__();self.input_width=14
        self.input_encoder=nn.Linear(14,hidden)
        self.conv1=SAGEConv(hidden,hidden,aggr='mean',project=True,normalize=False,root_weight=True,bias=True)
        self.conv2=SAGEConv(hidden,2,aggr='mean',project=True,normalize=False,root_weight=True,bias=True)
    def encode_input(self,x):
        require(x.ndim==2 and x.shape[1]==14,'Expected unchanged source14')
        return torch.tanh(self.input_encoder(x))
    def forward(self,x,edge_index):
        h=self.encode_input(x)
        return self.conv2(F.relu(self.conv1(h,edge_index)),edge_index)


### FrozenInteractionDecoder

Public pretrained conditional decoder: latent50 + source14 + relation6 ->14.

In [ ]:
# BASELINE_SOURCE_CELL: models.FrozenInteractionDecoder
class FrozenInteractionDecoder(nn.Module):
    """Public pretrained conditional decoder: latent50 + source14 + relation6 ->14."""
    def __init__(self):
        super().__init__();self.latent_size=50
        self.MLP=nn.Sequential()
        self.MLP.add_module('L0',nn.Linear(70,32));self.MLP.add_module('A0',nn.Tanh())
        self.MLP.add_module('L_out0',nn.Linear(32,14))
    def forward(self,z,source,relation):
        return torch.sigmoid(self.MLP(torch.cat([z,source,relation],dim=-1)))


### decoder_tensor_fingerprint

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: models.decoder_tensor_fingerprint
def decoder_tensor_fingerprint(state):
    digest=hashlib.sha256()
    for name in sorted(state):
        tensor=state[name].detach().cpu().contiguous()
        digest.update(name.encode());digest.update(str(tuple(tensor.shape)).encode());digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


### materialize_decoder

Restore visible numeric weights and verify their independent tensor fingerprint.

In [ ]:
# BASELINE_SOURCE_CELL: models.materialize_decoder
def materialize_decoder(output,reference):
    """Restore visible numeric weights and verify their independent tensor fingerprint."""
    state={name:torch.tensor(values,dtype=torch.float32) for name,values in reference['tensors'].items()}
    require(decoder_tensor_fingerprint(state)==reference['tensor_fingerprint'],'Inline public decoder tensors changed')
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    path=output/'public_decoder.pt'
    if path.exists():
        saved=torch.load(path,map_location='cpu',weights_only=True)
        require(decoder_tensor_fingerprint(saved)==reference['tensor_fingerprint'],'Cached decoder changed')
    else:atomic_torch_save(state,path)
    decoder=FrozenInteractionDecoder();decoder.load_state_dict(state,strict=True)
    decoder.eval();decoder.requires_grad_(False)
    return decoder


### author_row_normalize

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: models.author_row_normalize
def author_row_normalize(values):
    inverse=1/values.sum(dim=1);inverse[torch.isinf(inverse)]=0
    return values*inverse[:,None]


### save_meta_views

Cache public-code conditional interaction views; RNG/traversal are declared corrections.

In [ ]:
# BASELINE_SOURCE_CELL: models.save_meta_views
def save_meta_views(graph,records,output,decoder,role_seed,views):
    """Cache public-code conditional interaction views; RNG/traversal are declared corrections."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    generator=torch.Generator(device='cpu').manual_seed(role_seed+50000)
    files={}
    for record in records:
        batch=torch.load(record['path'],map_location='cpu',weights_only=True)
        name=Path(record['path']).stem
        for nt in ('EOA','CA'):
            source=graph[nt].x[batch['node_ids'][nt]].cpu()
            require(len(source)>0,'Meta-IFD requires both node types in each sampled batch')
            path=output/f'{name}_{nt}_views.npy'
            values=np.lib.format.open_memmap(path,mode='w+',dtype=np.float32,shape=(views,4,len(source),14))
            for view in range(views):
                interaction=0
                for rid,et in enumerate(graph.edge_types):
                    if et[0]!=nt:continue
                    for start in range(0,len(source),65536):
                        part=source[start:start+65536]
                        z=torch.randn((len(part),50),generator=generator)
                        condition=torch.zeros((len(part),6));condition[:,rid]=1
                        with torch.no_grad():generated=author_row_normalize(decoder(z,part,condition))
                        require(bool(torch.isfinite(generated).all()),'Nonfinite generated view')
                        values[view,interaction,start:start+len(part)]=generated.numpy()
                    interaction+=1
                require(interaction==3,'Expected three outgoing relation types')
                values[view,3]=source.numpy()
            values.flush();values._mmap.close();del values
            files[path.name]=sha256(path)
    return files


### build_model

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: models.build_model
def build_model(graph,model_name,config,target,statistics=None):
    if model_name=='gat':return SignedLogGAT(config['hidden'],statistics)
    if model_name=='sage':return InputEncodedSAGE(config['hidden'])
    require(model_name=='metaifd','Unknown neural model')
    return HMSL(config['hidden'],2,graph,config['views'],target,input_width=14)


### forward_batch

Only X14 or generated source14 views enter the classifier.

In [ ]:
# BASELINE_SOURCE_CELL: models.forward_batch
def forward_batch(model,model_name,graph,batch,view_dir,device):
    """Only X14 or generated source14 views enter the classifier."""
    ids=batch['node_ids'];edges={et:ix.to(device) for et,ix in batch['edge_index'].items()}
    if model_name=='metaifd':
        name=batch['batch_name']
        views={nt:torch.from_numpy(np.array(np.load(Path(view_dir)/f'{name}_{nt}_views.npy',mmap_mode='r'),copy=True)).to(device)
               for nt in ('EOA','CA')}
        logits,contrast=model(views,edges)
        return logits[:len(batch['seed_ids'])],contrast
    source={nt:graph[nt].x[ids[nt]].to(device) for nt in ('EOA','CA')}
    offset={'EOA':0,'CA':len(ids['EOA'])}
    homogeneous=[ix+torch.tensor([[offset[et[0]]],[offset[et[2]]]],device=device) for et,ix in edges.items()]
    logits=model(torch.cat([source['EOA'],source['CA']]),torch.cat(homogeneous,dim=1))
    begin=offset[batch['target_type']]
    return logits[begin:begin+len(batch['seed_ids'])],logits.new_zeros(())


## Implementation: metrics

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: metrics.imports
import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score


### classification_metrics

Shared implementation used by the explicit stages below.

In [ ]:
# BASELINE_SOURCE_CELL: metrics.classification_metrics
def classification_metrics(labels, scores, predictions):
    labels, scores, predictions = np.asarray(labels), np.asarray(scores), np.asarray(predictions)
    require(np.isfinite(scores).all(), 'Nonfinite scores')
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0,1]).ravel()
    return dict(macro_f1=float(f1_score(labels, predictions, average='macro', labels=[0,1], zero_division=0)),
                micro_f1=float(f1_score(labels, predictions, average='micro', zero_division=0)),
                binary_f1=float(f1_score(labels, predictions, zero_division=0)),
                precision=float(precision_score(labels, predictions, zero_division=0)),
                recall=float(recall_score(labels, predictions, zero_division=0)),
                accuracy=float(accuracy_score(labels, predictions)), fpr=float(fp/(fp+tn)) if fp+tn else None,
                average_precision=float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
                roc_auc=float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), n=len(labels))


## Implementation: study

Readable definitions only; training starts in the execution cells below.

In [ ]:
# BASELINE_SOURCE_CELL: study.imports
import gc
import hashlib
import itertools
from pathlib import Path
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from threadpoolctl import threadpool_limits
MODELS=('metaifd','gat','sage','lightgbm','lr','rf')
TABULAR=('lightgbm','lr','rf')


### fixed_configuration

One accepted-study configuration per model, chosen before this run; no grid.

In [ ]:
# BASELINE_SOURCE_CELL: study.fixed_configuration
def fixed_configuration(model_name):
    """One accepted-study configuration per model, chosen before this run; no grid."""
    configurations=dict(metaifd=dict(hidden=128,lr=.001,views=3,contrast_weight=.1),
        gat=dict(hidden=32,lr=.01),sage=dict(hidden=32,lr=.01),
        lightgbm=dict(n_estimators=300,lr=.01,num_leaves=31),
        lr=dict(C=10.,solver='liblinear',max_iter=5000),rf=dict(n_estimators=100,max_depth=10))
    require(model_name in configurations,'Only the six agreed models are supported')
    return configurations[model_name].copy()


### notebook_source_signature

Bind the literal executed scientific cells to this experiment identity.

In [ ]:
# BASELINE_SOURCE_CELL: study.notebook_source_signature
def notebook_source_signature():
    """Bind the literal executed scientific cells to this experiment identity."""
    shell=get_ipython()
    require(shell is not None,'Run the notebook with a normal IPython/Jupyter kernel')
    latest={}
    for source in shell.history_manager.input_hist_raw:
        first=source.splitlines()[0] if source.splitlines() else ''
        prefix='# BASELINE_SOURCE_CELL: '
        if first.startswith(prefix):latest[first[len(prefix):]]=hashlib.sha256(source.encode()).hexdigest()
    require(set(DEFINITION_KEYS)<=set(latest),'Run all definition cells first')
    return {key:latest[key] for key in DEFINITION_KEYS}


### open_study

Freeze masks, fixed config, code, reference data, RNG and runtime before training.

In [ ]:
# BASELINE_SOURCE_CELL: study.open_study
def open_study(graph,membership,data_manifest,job_dir,dataset,model_name,device,source_signature,
               decoder_reference=None,run_mode='full',cpu_threads=2):
    """Freeze masks, fixed config, code, reference data, RNG and runtime before training."""
    require(model_name in MODELS and run_mode in ('full','smoke'),'Invalid model or run mode')
    require(set(membership)=={'train','validation','test'},'Missing split role')
    torch.set_num_threads(cpu_threads)
    neural=model_name not in TABULAR
    require(neural or str(device)=='cpu','Tabular models must use CPU')
    protocol=dict(version='released_x14_native_fixed_v2',dataset=dataset,model=model_name,arm='X14',width=14,
        cohort='released_reference; retain all released labelled rows, including known Phish conflict',
        split='unchanged graph masks; one partition; no resplitting',
        fixed_config=fixed_configuration(model_name),configuration_source=(
            'public utils.py defaults, also used by accepted study; not a recovered paper-table winner' if model_name=='metaifd' else
            'accepted-study fixed settings; not recovered paper-table winning config'),
        model_seeds=[12,13,14,15,16] if run_mode=='full' else [12],run_mode=run_mode,
        hyperparameter_search=False,decision='p1 > 0.5; exact tie -> class0; no threshold search',
        preprocessing=('raw source14; StandardScaler fitted only to TRAIN roots' if model_name=='lr' else
                       'raw source14; no external scaler' if model_name in ('rf','lightgbm') else
                       'source14 signed-log1p then frozen z-score from unique typed TRAIN sampled nodes' if model_name=='gat' else
                       'raw source14; learned Linear14->hidden then tanh; mean SAGE project=True' if model_name=='sage' else
                       'public decoder generated normalized interaction views + unchanged source14'),
        max_epochs=(1000 if run_mode=='full' else 2) if neural else None,
        patience=50 if neural else None,weight_decay=.0005 if neural else None,
        checkpoint_rule='latest validation Macro-F1 tie; stop stale >50' if neural else 'one fitted estimator at fixed budget',
        fanouts=[100,100] if run_mode=='full' else [2,2],batch_size=512 if run_mode=='full' else 8,
        sampler_seeds=dict(train=1201,validation=1202,test=2301),
        sampler='actual PyG NeighborLoader directional; per-role cached draws shared across model seeds' if neural else 'none',
        author_commit='439c6daa936f67f19486be76642e690a2153d0c5',
        meta_loss='public-code unsquared L2 TripletMarginLoss mean, margin .3; differs from squared/sum paper equation',
        test_policy='freeze all five seed estimators/checkpoints before test cache or test inference',
        setting='static transductive; topology and unlabeled held-out features may be sampled as neighbors',
        inference='descriptive seed variation on one released split; not independent-split uncertainty',
        exact_paper_table_reproduction_claimed=False)
    protocol['backbone_source']={
        'gat':'audited phish_readable_gat / phish_gat_signedlog_exact_env211_v2',
        'sage':'audited phish_readable_graphsage / phish_graphsage_mean_dataset_exact_env211_v3',
        'metaifd':'pinned public HMSL and decoder'}.get(model_name,'accepted-study tabular baseline')
    protocol['architecture_status']='group implementation; not an exact original-paper GAT/SAGE reconstruction' if model_name in ('gat','sage') else 'see deviations'
    identity=dict(protocol=protocol,data=data_manifest,runtime=runtime_information(device),source=source_signature,
        decoder_reference=json_hash(decoder_reference) if decoder_reference else None,
        membership={role:frame.to_dict('records') for role,frame in membership.items()})
    job=Path(job_dir);job.mkdir(parents=True,exist_ok=True)
    path=job/'protocol.json'
    if path.exists():require(read_json(path)==identity,'Code/config/data/runtime changed; use a new output folder')
    else:write_json(path,identity)
    write_json(job/'split_report.json',data_manifest['split_report'])
    for role,part in membership.items():part.to_csv(job/(role+'_membership.csv'),index=False)
    return dict(graph=graph,membership=membership,protocol=protocol,identity=json_hash(identity),
                job=job,device=torch.device(device),decoder_reference=decoder_reference)


### prepare_role

Cache only requested roots/topology/views; test cache requires the model lock.

In [ ]:
# BASELINE_SOURCE_CELL: study.prepare_role
def prepare_role(study,role):
    """Cache only requested roots/topology/views; test cache requires the model lock."""
    require(role in ('train','validation','test'),'Unknown role')
    if role=='test':verify_model_lock(study)
    job=study['job'];root=job/'cache'/role;manifest_path=root/'manifest.json'
    if manifest_path.exists():
        manifest=read_json(manifest_path)
        require(manifest['identity']==study['identity'],'Cache belongs to another experiment')
        verify_files(root,manifest['files'])
        return [dict(path=str(root/'roots.npz'))] if study['protocol']['model'] in TABULAR else load_role_batches(root/'batches')
    root.mkdir(parents=True,exist_ok=True)
    graph=study['graph'];protocol=study['protocol'];target='EOA' if protocol['dataset']=='Phish' else 'CA'
    membership=study['membership'][role]
    if protocol['model'] in TABULAR:
        ids=membership.node_id.to_numpy(dtype=np.int64,copy=True)
        values=graph[target].x[torch.from_numpy(ids)].numpy().astype(np.float64)
        require(values.shape==(len(ids),14) and np.isfinite(values).all(),'Invalid source14')
        np.savez_compressed(root/'roots.npz',ids=ids,labels=membership.label.to_numpy(dtype=np.int64),source14=values)
        records=[dict(path=str(root/'roots.npz'))]
    else:
        records=save_pyg_role_batches(graph,membership,target,role,root/'batches',
            protocol['sampler_seeds'][role],protocol['fanouts'],protocol['batch_size'])
        if protocol['model']=='metaifd':
            require(study['decoder_reference'] is not None,'Missing public decoder numeric reference')
            decoder=materialize_decoder(job/'reference',study['decoder_reference'])
            save_meta_views(graph,records,root/'views',decoder,protocol['sampler_seeds'][role],protocol['fixed_config']['views'])
            del decoder
    files=[p for p in root.rglob('*') if p.is_file() and not p.name.endswith('.partial')]
    write_json(manifest_path,dict(identity=study['identity'],files=seal_files(root,files)))
    return records


### make_estimator

Match accepted-study tabular settings; scaler is part of the saved LR pipeline.

In [ ]:
# BASELINE_SOURCE_CELL: study.make_estimator
def make_estimator(model_name,config,seed,cpu_threads):
    """Match accepted-study tabular settings; scaler is part of the saved LR pipeline."""
    if model_name=='lr':
        return Pipeline([('scale',StandardScaler()),('estimator',LogisticRegression(
            C=config['C'],solver=config['solver'],max_iter=config['max_iter'],class_weight=None,random_state=seed))])
    if model_name=='rf':
        return RandomForestClassifier(n_estimators=config['n_estimators'],max_depth=config['max_depth'],
            max_features='sqrt',min_samples_leaf=1,bootstrap=True,class_weight=None,random_state=seed,n_jobs=cpu_threads)
    require(model_name=='lightgbm','Unknown tabular model')
    from lightgbm import LGBMClassifier
    return LGBMClassifier(objective='binary',n_estimators=config['n_estimators'],learning_rate=config['lr'],
        num_leaves=31,max_depth=-1,min_child_samples=20,subsample=1.,colsample_bytree=1.,
        class_weight=None,random_state=seed,n_jobs=cpu_threads,verbosity=-1)


### fit_gat_statistics

Fit signed-log mean/population SD on unique typed TRAIN neighbors, without labels.

In [ ]:
# BASELINE_SOURCE_CELL: study.fit_gat_statistics
def fit_gat_statistics(graph,records):
    """Fit signed-log mean/population SD on unique typed TRAIN neighbors, without labels."""
    by_type={'EOA':[],'CA':[]}
    for record in records:
        batch=torch.load(record['path'],map_location='cpu',weights_only=True)
        for nt in by_type:by_type[nt].append(batch['node_ids'][nt].numpy())
    ids={nt:np.unique(np.concatenate(parts)) for nt,parts in by_type.items()}
    total=torch.zeros(14,dtype=torch.float64);count=sum(len(i) for i in ids.values())
    require(count>0,'Empty GAT TRAIN fitting scope')
    for nt,index in ids.items():
        for start in range(0,len(index),100000):
            values=graph[nt].x[torch.from_numpy(index[start:start+100000])].double()
            values=values.sign()*values.abs().log1p()
            require(bool(torch.isfinite(values).all()),'Nonfinite GAT source14')
            total+=values.sum(0)
    mean=total/count;squared=torch.zeros(14,dtype=torch.float64)
    for nt,index in ids.items():
        for start in range(0,len(index),100000):
            values=graph[nt].x[torch.from_numpy(index[start:start+100000])].double()
            values=values.sign()*values.abs().log1p()
            squared+=(values-mean).square().sum(0)
    scale=(squared/count).sqrt();constant=scale<1e-12;scale[constant]=1.
    return dict(mean=mean.float().tolist(),scale=scale.float().tolist(),
        constant_columns=torch.where(constant)[0].tolist(),unique_train_nodes=count,
        counts_by_type={nt:len(i) for nt,i in ids.items()},labels_used=False,
        fitting_scope='unique (type,id) in TRAIN sampled batches, including unlabeled neighbors; no validation/test batch fitting',
        transform='sign(x)*log1p(abs(x)); population SD; no clipping; frozen float32 buffers',
        computation='chunked float64 mean then centered second pass; chunk summation may differ in rounding from full-tensor audit')


### tabular_predictions

Predict using the saved TRAIN-fitted preprocessing, without fitting any role.

In [ ]:
# BASELINE_SOURCE_CELL: study.tabular_predictions
def tabular_predictions(estimator,record):
    """Predict using the saved TRAIN-fitted preprocessing, without fitting any role."""
    with np.load(record['path'],allow_pickle=False) as data:
        values=data['source14'];labels=data['labels'];ids=data['ids']
    require(np.array_equal(estimator.classes_,[0,1]),'Unexpected estimator class order')
    with threadpool_limits(limits=torch.get_num_threads()):scores=estimator.predict_proba(values)[:,1]
    predictions=(scores>.5).astype(np.int64)
    metrics=classification_metrics(labels,scores,predictions)
    rows=pd.DataFrame(dict(node_id=ids,label=labels,score=scores,prediction=predictions))
    return metrics,rows


### run_neural_epoch

Cross-entropy plus the public contrastive loss; supervise sampled roots only.

In [ ]:
# BASELINE_SOURCE_CELL: study.run_neural_epoch
def run_neural_epoch(study,model,records,role,optimizer=None):
    """Cross-entropy plus the public contrastive loss; supervise sampled roots only."""
    model.train(optimizer is not None);rows=[];loss_sum=0.
    p=study['protocol']
    for record in records:
        batch=torch.load(record['path'],map_location='cpu',weights_only=True)
        batch['batch_name']=Path(record['path']).stem
        y=batch['seed_labels'].to(study['device'])
        with torch.set_grad_enabled(optimizer is not None):
            logits,contrast=forward_batch(model,p['model'],study['graph'],batch,study['job']/'cache'/role/'views',study['device'])
            loss=F.cross_entropy(logits,y)+p['fixed_config'].get('contrast_weight',0.)*contrast
            require(bool(torch.isfinite(logits).all() and torch.isfinite(loss)),'Nonfinite forward/loss')
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);loss.backward()
                require(all(v.grad is None or bool(torch.isfinite(v.grad).all()) for v in model.parameters()),'Nonfinite gradient')
                optimizer.step()
        scores=logits.detach().softmax(dim=1)[:,1].cpu().numpy()
        # Probability threshold is shared with tabular models, including exact float32 ties.
        predictions=(scores>.5).astype(np.int64)
        labels=y.cpu().numpy();loss_sum+=float(loss.detach().cpu())*len(labels)
        rows.extend(dict(node_id=int(i),label=int(y),score=float(s),prediction=int(pred)) for i,y,s,pred in
                    zip(batch['seed_ids'].tolist(),labels,scores,predictions))
    frame=pd.DataFrame(rows)
    require(len(frame)>0 and frame.node_id.is_unique,'Missing/duplicate neural root predictions')
    metrics=classification_metrics(frame.label,frame.score,frame.prediction);metrics['loss']=loss_sum/len(frame)
    return metrics,frame


### fit_one_seed

Train once at the fixed configuration; choose neural epochs with validation only.

In [ ]:
# BASELINE_SOURCE_CELL: study.fit_one_seed
def fit_one_seed(study,seed,train_records,validation_records):
    """Train once at the fixed configuration; choose neural epochs with validation only."""
    p=study['protocol'];job=study['job'];trial=job/'fits'/f'seed_{seed}';trial.mkdir(parents=True,exist_ok=True)
    completed=trial/'fit.json';identity=json_hash(dict(study=study['identity'],seed=seed,
        input_statistics=study.get('input_statistics')))
    if completed.exists():
        fit=read_json(completed);require(fit['identity']==identity,'Fit identity changed')
        verify_files(trial,fit['files']);return fit
    seed_everything(seed);started=time.perf_counter();captured=[]
    model_name=p['model'];target='EOA' if p['dataset']=='Phish' else 'CA';history=[]
    if model_name in TABULAR:
        estimator=make_estimator(model_name,p['fixed_config'],seed,torch.get_num_threads())
        with np.load(train_records[0]['path'],allow_pickle=False) as data:
            values=data['source14'];labels=data['labels']
        with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=torch.get_num_threads()):
            warnings.simplefilter('always');estimator.fit(values,labels)
        validation,_=tabular_predictions(estimator,validation_records[0])
        checkpoint=trial/'estimator.joblib';temporary=checkpoint.with_suffix('.partial')
        joblib.dump(estimator,temporary);temporary.replace(checkpoint)
        history=[dict(stage='fit_once',validation_macro_f1=validation['macro_f1'])]
        best_epoch=None;stopped_epoch=None;parameter_count=None
        detail=dict(fitted_features=int(values.shape[1]),training_rows=len(labels),
                    scaler_rows=int(estimator['scale'].n_samples_seen_) if model_name=='lr' else None)
        files=[checkpoint]
    else:
        if model_name=='gat':require(study.get('input_statistics') is not None,'Fit GAT TRAIN statistics before fitting models')
        model=build_model(study['graph'],model_name,p['fixed_config'],target,study.get('input_statistics')).to(study['device'])
        if model_name=='metaifd':model.reset_parameters()
        optimizer=torch.optim.Adam(model.parameters(),lr=p['fixed_config']['lr'],weight_decay=p['weight_decay'])
        best=-1.;stale=0;state=None;best_epoch=None
        for epoch in range(1,p['max_epochs']+1):
            train_metrics,_=run_neural_epoch(study,model,train_records,'train',optimizer)
            validation,_=run_neural_epoch(study,model,validation_records,'validation')
            improved=validation['macro_f1']>=best
            if improved:
                best=validation['macro_f1'];best_epoch=epoch;stale=0
                state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            else:stale+=1
            history.append(dict(epoch=epoch,train_loss=train_metrics['loss'],validation_macro_f1=validation['macro_f1'],
                                validation_loss=validation['loss'],best_epoch=best_epoch,stale=stale))
            if epoch==1 or epoch%25==0:print(model_name,'seed',seed,'epoch',epoch,'validation Macro-F1',validation['macro_f1'],flush=True)
            if stale>p['patience']:break
        stopped_epoch=epoch;require(state is not None,'No validation checkpoint')
        last=trial/'last_checkpoint.pt'
        atomic_torch_save(dict(state_dict=model.state_dict(),epoch=epoch,identity=identity,diagnostic_only=True),last)
        model.load_state_dict(state,strict=True)
        validation,_=run_neural_epoch(study,model,validation_records,'validation')
        require(abs(validation['macro_f1']-best)<1e-12,'Validation-best checkpoint replay mismatch')
        checkpoint=trial/'checkpoint.pt'
        atomic_torch_save(dict(state_dict=state,best_epoch=best_epoch,identity=identity),checkpoint)
        parameter_count=sum(v.numel() for v in model.parameters());detail={}
        files=[checkpoint,last];del model,optimizer,state;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    pd.DataFrame(history).to_csv(trial/'history.csv',index=False);files.append(trial/'history.csv')
    fit=dict(identity=identity,seed=seed,model=model_name,config=p['fixed_config'],input_width=14,
             validation=validation,best_epoch=best_epoch,stopped_epoch=stopped_epoch,
             parameter_count=parameter_count,training_seconds=time.perf_counter()-started,
             warnings=[str(w.message) for w in captured],detail=detail,test_scored=False,
             checkpoint=checkpoint.name,files=seal_files(trial,files))
    write_json(completed,fit);return fit


### fit_and_lock

Save all requested seed fits before allowing any test cache or score.

In [ ]:
# BASELINE_SOURCE_CELL: study.fit_and_lock
def fit_and_lock(study):
    """Save all requested seed fits before allowing any test cache or score."""
    if (study['job']/'model_lock.json').exists():return verify_model_lock(study)
    require(not (study['job']/'cache/test').exists(),'Test cache exists before model lock')
    train_records=prepare_role(study,'train')
    if study['protocol']['model']=='gat':
        statistics=fit_gat_statistics(study['graph'],train_records)
        path=study['job']/'input_statistics.json'
        if path.exists():require(read_json(path)==statistics,'Frozen GAT statistics changed')
        else:write_json(path,statistics)
        study['input_statistics']=statistics
    validation_records=prepare_role(study,'validation')
    fits=[]
    for seed in study['protocol']['model_seeds']:
        fits.append(fit_one_seed(study,seed,train_records,validation_records))
    files=[p for p in (study['job']/'fits').rglob('*') if p.is_file() and not p.name.endswith('.partial')]
    files += [study['job']/'protocol.json',study['job']/'cache/train/manifest.json',study['job']/'cache/validation/manifest.json']
    if (study['job']/'input_statistics.json').exists():files.append(study['job']/'input_statistics.json')
    lock=dict(identity=study['identity'],seeds=study['protocol']['model_seeds'],fits=fits,
              files=seal_files(study['job'],files),test_scored=False)
    write_json(study['job']/'model_lock.json',lock)
    return verify_model_lock(study)


### verify_model_lock

Verify the full set of fixed seed estimators/checkpoints and their training caches.

In [ ]:
# BASELINE_SOURCE_CELL: study.verify_model_lock
def verify_model_lock(study):
    """Verify the full set of fixed seed estimators/checkpoints and their training caches."""
    path=study['job']/'model_lock.json';require(path.exists(),'Fit and lock every seed before test')
    lock=read_json(path);require(lock['identity']==study['identity'],'Lock identity mismatch')
    require(lock['seeds']==study['protocol']['model_seeds'] and [f['seed'] for f in lock['fits']]==lock['seeds'],
            'Incomplete or duplicated seed fits')
    verify_files(study['job'],lock['files'])
    for role in ('train','validation'):
        root=study['job']/'cache'/role;verify_files(root,read_json(root/'manifest.json')['files'])
    return lock


### evaluate_locked

Restore saved fits and predict; this phase never fits models or selects settings.

In [ ]:
# BASELINE_SOURCE_CELL: study.evaluate_locked
def evaluate_locked(study):
    """Restore saved fits and predict; this phase never fits models or selects settings."""
    lock=verify_model_lock(study);job=study['job'];result_path=job/'RESULT_MANIFEST.json'
    if result_path.exists():
        result=read_json(result_path)
        require(result['identity']==study['identity'] and result['model_lock_sha256']==sha256(job/'model_lock.json'),'Evaluation identity changed')
        verify_files(job,result['files'])
        return pd.read_csv(job/'metrics.csv')
    records={role:prepare_role(study,role) for role in ('train','validation','test')}
    metrics=[];prediction_dir=job/'predictions';prediction_dir.mkdir(exist_ok=True)
    p=study['protocol'];target='EOA' if p['dataset']=='Phish' else 'CA'
    for fit in lock['fits']:
        trial=job/'fits'/f"seed_{fit['seed']}";verify_files(trial,fit['files'])
        if p['model'] in TABULAR:model=joblib.load(trial/fit['checkpoint'])
        else:
            saved=torch.load(trial/fit['checkpoint'],map_location='cpu',weights_only=True)
            require(saved['identity']==fit['identity'],'Checkpoint identity mismatch')
            model=build_model(study['graph'],p['model'],p['fixed_config'],target).to(study['device'])
            model.load_state_dict(saved['state_dict'],strict=True);model.eval()
        for role in ('train','validation','test'):
            scores,frame=(tabular_predictions(model,records[role][0]) if p['model'] in TABULAR else
                          run_neural_epoch(study,model,records[role],role))
            expected=study['membership'][role].set_index('node_id')
            require(frame.node_id.is_unique and set(frame.node_id)==set(expected.index),'Prediction split coverage mismatch')
            require(np.array_equal(frame.label.to_numpy(),expected.loc[frame.node_id,'label'].to_numpy()),'Prediction label mismatch')
            if role=='validation':require(abs(scores['macro_f1']-fit['validation']['macro_f1'])<1e-12,'Saved validation replay changed')
            frame['address']=expected.loc[frame.node_id,'address'].to_numpy()
            frame.to_csv(prediction_dir/f"seed_{fit['seed']}_{role}.csv",index=False)
            metrics.append(dict(dataset=p['dataset'],model=p['model'],arm='X14',seed=fit['seed'],role=role,**scores))
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    frame=pd.DataFrame(metrics);frame.to_csv(job/'metrics.csv',index=False)
    keys=['macro_f1','micro_f1','binary_f1','precision','recall','accuracy','fpr','average_precision','roc_auc']
    summary=[]
    for role,part in frame.groupby('role'):
        for key in keys:
            summary.append(dict(dataset=p['dataset'],model=p['model'],role=role,metric=key,
                mean=float(part[key].mean()),sd=float(part[key].std(ddof=1)) if len(part)>1 else None,
                seeds=len(part),unit='fraction; descriptive seed SD on one fixed split'))
    pd.DataFrame(summary).to_csv(job/'summary.csv',index=False)
    files=[job/'metrics.csv',job/'summary.csv']+list(prediction_dir.glob('*.csv'))
    write_json(result_path,dict(identity=study['identity'],model_lock_sha256=sha256(job/'model_lock.json'),
               files=seal_files(job,files),full_benchmark=p['run_mode']=='full',test_used_for_selection=False))
    return frame


### export_results

Package fitted artifacts and audit files, excluding large sampled topology/view caches.

In [ ]:
# BASELINE_SOURCE_CELL: study.export_results
def export_results(study,output):
    """Package fitted artifacts and audit files, excluding large sampled topology/view caches."""
    import zipfile
    verify_model_lock(study)
    result=read_json(study['job']/'RESULT_MANIFEST.json');verify_files(study['job'],result['files'])
    output=Path(output)
    with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in study['job'].rglob('*'):
            if not path.is_file() or path.name.endswith('.partial'):continue
            relative=path.relative_to(study['job'])
            if relative.parts[0]=='cache':
                if path.name not in ('manifest.json','index.json'):continue
            archive.write(path,relative.as_posix())
    return output


## Plain reference data

Audited labelled-root addresses, file checksums and (Meta-IFD only) public pretrained decoder tensor values. These are ordinary data, not encoded Python or fitted baseline weights. Expand for inspection.

In [ ]:
REFERENCE={'graph_sha256': '61cdeba7b59d5ed041f07821b9f2fe0ad86cb1a36cd2d87b2740af938719e8f4',
 'ids': [0,
         1,
         2,
         3,
         4,
         5,
         6,
         7,
         8,
         9,
         10,
         11,
         12,
         13,
         14,
         15,
         16,
         17,
         18,
         19,
         20,
         21,
         22,
         23,
         24,
         25,
         26,
         27,
         28,
         29,
         30,
         31,
         32,
         33,
         34,
         35,
         36,
         37,
         38,
         39,
         40,
         41,
         42,
         43,
         44,
         45,
         46,
         47,
         48,
         49,
         50,
         51,
         52,
         53,
         54,
         55,
         56,
         57,
         58,
         59,
         60,
         61,
         62,
         63,
         64,
         65,
         66,
         67,
         68,
         69,
         70,
         71,
         72,
         73,
         74,
         75,
         76,
         77,
         78,
         79,
         80,
         81,
         82,
         83,
         84,
         85,
         86,
         87,
         88,
         89,
         90,
         91,
         92,
         93,
         94,
         95,
         96,
         97,
         98,
         99,
         100,
         101,
         102,
         103,
         104,
         105,
         106,
         107,
         108,
         109,
         110,
         111,
         112,
         113,
         114,
         115,
         116,
         117,
         118,
         119,
         120,
         121,
         122,
         123,
         124,
         125,
         126,
         127,
         128,
         129,
         130,
         131,
         132,
         133,
         134,
         135,
         136,
         137,
         138,
         139,
         140,
         141,
         142,
         143,
         144,
         145,
         146,
         147,
         148,
         149,
         150,
         151,
         152,
         153,
         154,
         155,
         156,
         157,
         158,
         159,
         160,
         161,
         162,
         163,
         164,
         165,
         166,
         167,
         168,
         169,
         170,
         171,
         172,
         173,
         174,
         175,
         176,
         177,
         178,
         179,
         180,
         181,
         182,
         183,
         184,
         185,
         186,
         187,
         188,
         189,
         190,
         191,
         192,
         193,
         194,
         195,
         196,
         197,
         198,
         199,
         200,
         201,
         202,
         203,
         204,
         205,
         206,
         207,
         208,
         209,
         210,
         211,
         212,
         213,
         214,
         215,
         216,
         217,
         218,
         219,
         220,
         221,
         222,
         223,
         224,
         225,
         226,
         227,
         228,
         229,
         230,
         231,
         232,
         233,
         234,
         235,
         236,
         237,
         238,
         239,
         240,
         241,
         242,
         243,
         244,
         245,
         246,
         247,
         248,
         249,
         250,
         251,
         252,
         253,
         254,
         255,
         256,
         257,
         258,
         259,
         260,
         261,
         262,
         263,
         264,
         265,
         266,
         267,
         268,
         269,
         270,
         271,
         272,
         273,
         274,
         275,
         276,
         277,
         278,
         279,
         280,
         281,
         282,
         283,
         284,
         285,
         286,
         287,
         288,
         289,
         290,
         291,
         292,
         293,
         294,
         295,
         296,
         297,
         298,
         299,
         300,
         301,
         302,
         303,
         304,
         305,
         306,
         307,
         308,
         309,
         310,
         311,
         312,
         313,
         314,
         315,
         316,
         317,
         318,
         319,
         320,
         321,
         322,
         323,
         324,
         325,
         326,
         327,
         328,
         329,
         330,
         331,
         332,
         333,
         334,
         335,
         336,
         337,
         338,
         339,
         340,
         341,
         342,
         343,
         344,
         345,
         346,
         347,
         348,
         349,
         350,
         351,
         352,
         353,
         354,
         355,
         356,
         357,
         358,
         359,
         360,
         361,
         362,
         363,
         364,
         365,
         366,
         367,
         368,
         369,
         370,
         371,
         372,
         373,
         374,
         375,
         376,
         377,
         378,
         379,
         380,
         381,
         382,
         383,
         384,
         385,
         386,
         387,
         388,
         389,
         390,
         391,
         392,
         393,
         394,
         395,
         396,
         397,
         398,
         399,
         400,
         401,
         402,
         403,
         404,
         405,
         406,
         407,
         408,
         409,
         410,
         411,
         412,
         413,
         414,
         415,
         416,
         417,
         418,
         419,
         420,
         421,
         422,
         423,
         424,
         425,
         426,
         427,
         428,
         429,
         430,
         431,
         432,
         433,
         434,
         435,
         436,
         437,
         438,
         439,
         440,
         441,
         442,
         443,
         444,
         445,
         446,
         447,
         448,
         449,
         450,
         451,
         452,
         453,
         454,
         455,
         456,
         457,
         458,
         459,
         460,
         461,
         462,
         463,
         464,
         465,
         466,
         467,
         468,
         469,
         470,
         471,
         472,
         473,
         474,
         475,
         476,
         477,
         478,
         479,
         480,
         481,
         482,
         483,
         484,
         485,
         486,
         487,
         488,
         489,
         490,
         491,
         492,
         493,
         494,
         495,
         496,
         497,
         498,
         499,
         500,
         501,
         502,
         503,
         504,
         505,
         506,
         507,
         508,
         509,
         510,
         511,
         512,
         513,
         514,
         515,
         516,
         517,
         518,
         519,
         520,
         521,
         522,
         523,
         524,
         525,
         526,
         527,
         528,
         529,
         530,
         531,
         532,
         533,
         534,
         535,
         536,
         537,
         538,
         539,
         540,
         541,
         542,
         543,
         544,
         545,
         546,
         547,
         548,
         549,
         550,
         551,
         552,
         553,
         554,
         555,
         556,
         557,
         558,
         559,
         560,
         561,
         562,
         563,
         564,
         565,
         566,
         567,
         568,
         569,
         570,
         571,
         572,
         573,
         574,
         575,
         576,
         577,
         578,
         579,
         580,
         581,
         582,
         583,
         584,
         585,
         586,
         587,
         588,
         589,
         590,
         591,
         592,
         593,
         594,
         595,
         596,
         597,
         598,
         599,
         600,
         601,
         602,
         603,
         604,
         605,
         606,
         607,
         608,
         609,
         610,
         611,
         612,
         613,
         614,
         615,
         616,
         617,
         618,
         619,
         620,
         621,
         622,
         623,
         624,
         625,
         626,
         627,
         628,
         629,
         630,
         631,
         632,
         633,
         634,
         635,
         636,
         637,
         638,
         639,
         640,
         641,
         642,
         643,
         644,
         645,
         646,
         647,
         648,
         649,
         650,
         651,
         652,
         653,
         654,
         655,
         656,
         657,
         658,
         659,
         660,
         661,
         662,
         663,
         664,
         665,
         666,
         667,
         668,
         669,
         670,
         671,
         672,
         673,
         674,
         675,
         676,
         677,
         678,
         679,
         680,
         681,
         682,
         683,
         684,
         685,
         686,
         687,
         688,
         689,
         690,
         691,
         692,
         693,
         694,
         695,
         696,
         697,
         698,
         699,
         700,
         701,
         702,
         703,
         704,
         705,
         706,
         707,
         708,
         709,
         710,
         711,
         712,
         713,
         714,
         715,
         716,
         717,
         718,
         719,
         720,
         721,
         722,
         723,
         724,
         725,
         726,
         727,
         728,
         729,
         730,
         731,
         732,
         733,
         734,
         735,
         736,
         737,
         738,
         739,
         740,
         741,
         742,
         743,
         744,
         745,
         746,
         747,
         748,
         749,
         750,
         751,
         752,
         753,
         754,
         755,
         756,
         757,
         758,
         759,
         760,
         761,
         762,
         763,
         764,
         765,
         766,
         767,
         768,
         769,
         770,
         771,
         772,
         773,
         774,
         775,
         776,
         777,
         778,
         779,
         780,
         781,
         782,
         783,
         784,
         785,
         786,
         787,
         788,
         789,
         790,
         791,
         792,
         793,
         794,
         795,
         796,
         797,
         798,
         799,
         800,
         801,
         802,
         803,
         804,
         805,
         806,
         807,
         808,
         809,
         810,
         811,
         812,
         813,
         814,
         815,
         816,
         817,
         818,
         819,
         820,
         821,
         822,
         823,
         824,
         825,
         826,
         827,
         828,
         829,
         830,
         831,
         832,
         833,
         834,
         835,
         836,
         837,
         838,
         839,
         840,
         841,
         842,
         843,
         844,
         845,
         846,
         847,
         848,
         849,
         850,
         851,
         852,
         853,
         854,
         855,
         856,
         857,
         858,
         859,
         860,
         861,
         862,
         863,
         864,
         865,
         866,
         867,
         868,
         869,
         870,
         871,
         872,
         873,
         874,
         875,
         876,
         877,
         878,
         879,
         880,
         881,
         882,
         883,
         884,
         885,
         886,
         887,
         888,
         889,
         890,
         891,
         892,
         893,
         894,
         895,
         896,
         897,
         898,
         899,
         900,
         901,
         902,
         903,
         904,
         905,
         906,
         907,
         908,
         909,
         910,
         911,
         912,
         913,
         914,
         915,
         916,
         917,
         918,
         919,
         920,
         921,
         922,
         923,
         924,
         925,
         926,
         927,
         928,
         929,
         930,
         931,
         932,
         933,
         934,
         935,
         936,
         937,
         938,
         939,
         940,
         941,
         942,
         943,
         944,
         945,
         946,
         947,
         948,
         949,
         950,
         951,
         952,
         953,
         954,
         955,
         956,
         957,
         958,
         959,
         960,
         961,
         962,
         963,
         964,
         965,
         966,
         967,
         968,
         969,
         970,
         971,
         972,
         973,
         974,
         975,
         976,
         977,
         978,
         979,
         980,
         981,
         982,
         983,
         984,
         985,
         986,
         987,
         988,
         989,
         990,
         991,
         992,
         993,
         994,
         995,
         996,
         997,
         998,
         999,
         1000,
         1001,
         1002,
         1003,
         1004,
         1005,
         1006,
         1007,
         1008,
         1009,
         1010,
         1011,
         1012,
         1013,
         1014,
         1015,
         1016,
         1017,
         1018,
         1019,
         1020,
         1021,
         1022,
         1023,
         1024,
         1025,
         1026,
         1027,
         1028,
         1029,
         1030,
         1031,
         1032,
         1033,
         1034,
         1035,
         1036,
         1037,
         1038,
         1039,
         1040,
         1041,
         1042,
         1043,
         1044,
         1045,
         1046,
         1047,
         1048,
         1049,
         1050,
         1051,
         1052,
         1053,
         1054,
         1055,
         1056,
         1057,
         1058,
         1059,
         1060,
         1061,
         1062,
         1063,
         1064,
         1065,
         1066,
         1067,
         1068,
         1069,
         1070,
         1071,
         1072,
         1073,
         1074,
         1075,
         1076,
         1077,
         1078,
         1079,
         1080,
         1081,
         1082,
         1083,
         1084,
         1085,
         1086,
         1087,
         1088,
         1089,
         1090,
         1091,
         1092,
         1093,
         1094,
         1095,
         1096,
         1097,
         1098,
         1099,
         1100,
         1101,
         1102,
         1103,
         1104,
         1105,
         1106,
         1107,
         1108,
         1109,
         1110,
         1111,
         1112,
         1113,
         1114,
         1115,
         1116,
         1117,
         1118,
         1119,
         1120,
         1121,
         1122,
         1123,
         1124,
         1125,
         1126,
         1127,
         1128,
         1129,
         1130,
         1131,
         1132,
         1133,
         1134,
         1135,
         1136,
         1137,
         1138,
         1139,
         1140,
         1141,
         1142,
         1143,
         1144,
         1145,
         1146,
         1147,
         1148,
         1149,
         1150,
         1151,
         1152,
         1153,
         1154,
         1155,
         1156,
         1157,
         1158,
         1159,
         1160,
         1161,
         1162,
         1163,
         1164,
         1165,
         1166,
         1167,
         1168,
         1169,
         1170,
         1171,
         1172,
         1173,
         1174,
         1175,
         1176,
         1177,
         1178,
         1179,
         1180,
         1181,
         1182,
         1183,
         1184,
         1185,
         1186,
         1187,
         1188,
         1189,
         1190,
         1191,
         1192,
         1193,
         1194,
         1195,
         1196,
         1197,
         1198,
         1199,
         1200,
         1201,
         1202,
         1203,
         1204,
         1205,
         1206,
         1207,
         1208,
         1209,
         1210,
         1211,
         1212,
         1213,
         1214,
         1215,
         1216,
         1217,
         1218,
         1219,
         1220,
         1221,
         1222,
         1223,
         1224,
         1225,
         1226,
         1227,
         1228,
         1229,
         1230,
         1231,
         1232,
         1233,
         1234,
         1235,
         1236,
         1237,
         1238,
         1239,
         1240,
         1241,
         1242,
         1243,
         1244,
         1245,
         1246,
         1247,
         1248,
         1249,
         1250,
         1251,
         1252,
         1253,
         1254,
         1255,
         1256,
         1257,
         1258,
         1259,
         1260,
         1261,
         1262,
         1263,
         1264,
         1265,
         1266,
         1267,
         1268,
         1269,
         1270,
         1271,
         1272,
         1273,
         1274,
         1275,
         1276,
         1277,
         1278,
         1279,
         1280,
         1281,
         1282,
         1283,
         1284,
         1285,
         1286,
         1287,
         1288,
         1289,
         1290,
         1291,
         1292,
         1293,
         1294,
         1295,
         1296,
         1297,
         1298,
         1299,
         1300,
         1301,
         1302,
         1303,
         1304,
         1305,
         1306,
         1307,
         1308,
         1309,
         1310,
         1311,
         1312,
         1313,
         1314,
         1315,
         1316,
         1317,
         1318,
         1319,
         1320,
         1321,
         1322,
         1323,
         1324,
         1325,
         1326,
         1327,
         1328,
         1329,
         1330,
         1331,
         1332,
         1333,
         1334,
         1335,
         1336,
         1337,
         1338,
         1339,
         1340,
         1341],
 'labels': [1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0],
 'addresses': ['...'],
 'raw_members': {'Meta-IFD data/Ponzi_unbalance/CA_labeled.csv': {'sha256': 'd9deb51acc377d2a7fc1c04656f0d90a640596684d1cdc14735a0a54be9d85bb',
                                                                  'bytes': 240033},
                 'Meta-IFD data/Ponzi_unbalance/CA_unlabeled.csv': {'sha256': 'a2a2ff2b3a4486ba1b457579b374c6307f0a8654173ac1b63c9475711eb624ef',
                                                                    'bytes': 343178752},
                 'Meta-IFD data/Ponzi_unbalance/EOA.csv': {'sha256': '5299ec5d2fe62aaa1d6347416439c95dc044066bd2dc69c735f1282558a3efef',
                                                           'bytes': 1209674221},
                 'Meta-IFD data/Ponzi_unbalance/edge/call_ca_ca.csv': {'sha256': 'd44dcce79088a9aa0c2632b65ad9a6df55e54a784245f7698750b00002c41f76',
                                                                       'bytes': 457011781},
                 'Meta-IFD data/Ponzi_unbalance/edge/call_eoa_ca.csv': {'sha256': 'af18891948b53304ca68fa678731deb583a3783080d3ce632847a87e047cefc0',
                                                                        'bytes': 2397107086},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_ca_ca.csv': {'sha256': '65898127ae2286d9725b1f2c9f550ca0fb4ffaf30911b9c8263ef46271029d50',
                                                                        'bytes': 59202487},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_ca_eoa.csv': {'sha256': '0386b1967a814b7049176c028f1a80b590deff9fc227891b7194a1890e561992',
                                                                         'bytes': 241532233},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_eoa_ca.csv': {'sha256': '12c8465d0b4e84748f95e819fec10a3af56d055007eaabc58498581bf8590f49',
                                                                         'bytes': 325508430},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_eoa_eoa.csv': {'sha256': '1e8040229071cc700d13db38980910ec92840514078c89bc342bc70e26117f41',
                                                                          'bytes': 3215691170},
                 'Meta-IFD data/node_hete_EOA.csv': {'sha256': '0a0a2a601e81f8972622c4bdfdd81f070bebc57b65051303aa46a7d8ab52bcfb',
                                                     'bytes': 3422390643}}}

DECODER_REFERENCE=None


In [ ]:
DEFINITION_KEYS=['utilities.imports',
 'utilities.require',
 'utilities.sha256',
 'utilities.json_hash',
 'utilities.write_json',
 'utilities.read_json',
 'utilities.seed_everything',
 'utilities.runtime_information',
 'utilities.seal_files',
 'utilities.verify_files',
 'utilities.atomic_torch_save',
 'data.imports',
 'data.resolve_input',
 'data.open_raw_member',
 'data.obtain_graph',
 'data.released_membership',
 'data.prepare_release',
 'sampling.imports',
 'sampling.save_pyg_role_batches',
 'sampling.load_role_batches',
 'attention_conv.imports',
 'attention_conv.my_conv',
 'models_meta.imports',
 'models_meta.group',
 'models_meta.TripletLoss',
 'models_meta.HMSL',
 'models.imports',
 'models.SignedLogGAT',
 'models.InputEncodedSAGE',
 'models.FrozenInteractionDecoder',
 'models.decoder_tensor_fingerprint',
 'models.materialize_decoder',
 'models.author_row_normalize',
 'models.save_meta_views',
 'models.build_model',
 'models.forward_batch',
 'metrics.imports',
 'metrics.classification_metrics',
 'study.imports',
 'study.fixed_configuration',
 'study.notebook_source_signature',
 'study.open_study',
 'study.prepare_role',
 'study.make_estimator',
 'study.fit_gat_statistics',
 'study.tabular_predictions',
 'study.run_neural_epoch',
 'study.fit_one_seed',
 'study.fit_and_lock',
 'study.verify_model_lock',
 'study.evaluate_locked',
 'study.export_results']


## 1. Verify the release and preserve its split

Hash-check the graph before loading it, verify the raw labelled table and bind the audited root mapping. Retain the released masks and print address intersections. No feature extension or raw edge restoration is performed.

In [ ]:
SOURCE_SIGNATURE=notebook_source_signature()
RAW_SOURCE=resolve_input(DATA_RAW,DATASET,'raw')
PROCESSED_SOURCE=resolve_input(DATA_PHISH,DATASET,'processed')
print('Raw:',RAW_SOURCE,'| processed:',PROCESSED_SOURCE)
graph,membership,data_manifest=prepare_release(RAW_SOURCE,PROCESSED_SOURCE,DATA_OUTPUT,DATASET,REFERENCE)
display(pd.DataFrame(data_manifest['split_report']['sizes'],index=['roots']))
print('Address intersections:',data_manifest['split_report']['address_intersections'])


## 2. Freeze the fixed protocol

All settings, seeds, split IDs, source cells and runtime versions are part of the run identity. Reusing a folder after a change raises an error. No validation-based hyperparameter search occurs.

In [ ]:
study=open_study(graph,membership,data_manifest,JOB_DIR,DATASET,MODEL,DEVICE,SOURCE_SIGNATURE,
                 DECODER_REFERENCE,RUN_MODE,CPU_THREADS)
print('Fixed configuration:',study['protocol']['fixed_config'])
print('Model seeds:',study['protocol']['model_seeds'])
print('Preprocessing:',study['protocol']['preprocessing'])


## 3. Train and lock all five model seeds

LR fits its scaler only to TRAIN roots. RF/LightGBM fit raw source14. GAT fits signed-log/z-score only on unique typed nodes in TRAIN sampled batches. GraphSAGE uses raw source14 with a learned Linear/tanh encoder and project=True mean layers. All neural models use latest validation-best ties and stop after more than50stale epochs. Every requested estimator/checkpoint is saved and hash-locked before test sampling or test predictions. Completed fits resume without refitting.

In [ ]:
if notebook_source_signature()!=SOURCE_SIGNATURE:raise ValueError('Definitions changed during the study')
model_lock=fit_and_lock(study)
display(pd.DataFrame([dict(seed=f['seed'],validation_macro_f1=f['validation']['macro_f1'],
                          best_epoch=f['best_epoch'],training_seconds=f['training_seconds']) for f in model_lock['fits']]))
print('All requested fits are locked; no test scoring occurred in this training phase.')


## 4. Evaluate saved fits on the original test split

Restore the frozen estimators/checkpoints; never refit or change thresholds. Save per-address probabilities and predictions for TRAIN, validation and TEST. Replay validation to verify the loaded checkpoint. A completed rerun verifies saved predictions instead of performing new inference.

In [ ]:
if notebook_source_signature()!=SOURCE_SIGNATURE:raise ValueError('Definitions changed after model lock')
metrics=evaluate_locked(study)
display(metrics.loc[metrics.role=='test',['dataset','model','seed','macro_f1','micro_f1','average_precision','fpr']])
display(pd.read_csv(JOB_DIR/'summary.csv').query("role == 'test' and metric == 'macro_f1'"))


## 5. Download the baseline audit bundle

Keep the results ZIP containing fitted weights/estimators, protocol, split memberships, predictions, metrics and hashes. Large sampled batches/views are excluded; retain WORK_ROOT separately for exact cached resume. Scores are descriptive across five model seeds on one split, not20 independent partitions.

In [ ]:
archive=OUTPUT_ROOT/(DATASET+'_'+MODEL+'_'+RUN_MODE+'_native_v2_released_X14_results.zip')
export_results(study,archive)
print('COMPLETE. Download:',archive)
